# Langfuse Showcase — local self-host, Anthropic Claude

End-to-end demonstration of Langfuse against a small RAG-style pipeline. Four scenarios, each producing a distinct view in the Langfuse UI:

1. **Single LLM call** — basic trace primitive (input, output, latency, tokens, cost)
2. **Multi-step pipeline** — trace hierarchy (parent + children) over a retrieve-then-generate flow
3. **Dataset + experiment** — curated test set, run the pipeline against it, view results side by side
4. **LLM-as-judge evaluator** — faithfulness scoring attached to each trace and aggregated at the experiment level

Each scenario takes ~20 lines of code and produces something visually distinct in the UI — useful for video capture.

## Step 1 — Run Langfuse locally

From this project directory, in a terminal:

```bash
./setup.sh
```

This clones the official Langfuse repo into `./langfuse` and brings up the stack via `docker compose`. When it reports ready, open http://localhost:3000 — sign up (local account, no email gating on self-host), create an organization and a project.

Go to **Settings → API Keys** in your project and create a new key pair. Copy the public key (`pk-lf-...`) and secret key (`sk-lf-...`).

## Step 2 — Environment variables

Copy `.env.example` to `.env` in this directory and fill in your keys:

```bash
cp .env.example .env
# then edit .env
```

The cell below loads it via `python-dotenv`. The `.env` file is git-ignored so secrets stay local.

In [1]:
# %pip install -q langfuse anthropic opentelemetry-instrumentation-anthropic python-dotenv

import os
from dotenv import load_dotenv

# Loads LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST, ANTHROPIC_API_KEY from .env
load_dotenv()

# Sanity-check the required variables are present.
for var in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL", "ANTHROPIC_API_KEY"):
    assert os.environ.get(var), f"{var} missing — add it to .env"

In [2]:
from langfuse import get_client, observe
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
from anthropic import Anthropic

# Auto-instrument every Anthropic SDK call as a Langfuse generation.
# After this line, any `client.messages.create(...)` is tracked automatically:
# the input, output, model, token counts, and cost all flow into Langfuse.
AnthropicInstrumentor().instrument()

langfuse = get_client()
client   = Anthropic()

# Verify the local Langfuse instance is reachable and the keys are valid.
assert langfuse.auth_check(), "Langfuse auth failed — check keys and host."
print("Langfuse client authenticated against", os.environ["LANGFUSE_BASE_URL"])

Langfuse client authenticated against http://localhost:3000


## Model choice

All scenarios below use a single Claude model for both the system being evaluated and the LLM-as-judge. In a real setup you'd use a different model family for the judge to reduce self-preference bias — that point is covered in the deck.

In [3]:
MODEL = "claude-sonnet-4-5"   # swap for whichever model your API key has access to

---

## Scenario 1 — Single call, basic trace

One Anthropic call. No decorators, no orchestration. The `AnthropicInstrumentor` we set up above is already producing a trace for every API call — this cell demonstrates that the default instrumentation is enough to start.

**What to look for in the UI:** Open the **Tracing → Traces** view in Langfuse. A new trace appears with the prompt as input, the response as output, model name, latency, input/output token counts, and computed cost.

In [4]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "In two sentences, what is observability in software systems?"}
    ],
)
print(response.content[0].text)

# Flush ensures the trace reaches Langfuse before the notebook moves on.
# In long-running services this is automatic; in notebooks it's worth calling explicitly.
langfuse.flush()

Observability in software systems is the ability to understand and measure the internal state of a system based on the data it produces, such as logs, metrics, and traces. It enables engineers to ask arbitrary questions about system behavior and diagnose issues without needing to predict every possible failure mode in advance.


## Scenario 2 — Multi-step trace: retrieve, then generate

A small retrieve-then-generate pipeline standing in for a real RAG system. The point isn't the retrieval logic (it's keyword-matched against a fixed dictionary); the point is that wrapping each step in `@observe` produces a *trace hierarchy* in Langfuse — a parent span with child operations nested under it.

**What to look for in the UI:** A single trace named `rag_pipeline` containing two child observations: `retrieve` (the lookup) and `generate` (the Anthropic call). Click into the trace to see the full tree, with timing and token counts at each level.

In [5]:
# A tiny knowledge base standing in for a vector store.
KNOWLEDGE_BASE = {
    "vpn_password":   "To reset your VPN password, visit selfservice.example.com and click 'Reset VPN'. The new password is emailed to your registered address.",
    "laptop_refresh": "Standard laptops are refreshed every four years. Specialized engineering workstations are refreshed every three years.",
    "expense_report": "Submit expense reports through the Concur portal at concur.example.com. Reports must be filed within 30 days of the expense.",
    "vacation":       "Full-time employees accrue 20 days of paid time off per year, increasing to 25 days after 5 years of service.",
    "guest_wifi":     "Guest WiFi credentials are available at the reception desk. The network is named 'Guest-Network' and credentials rotate weekly.",
}

@observe()
def retrieve(query: str) -> str:
    """Keyword-match the query against the knowledge base."""
    q = query.lower()
    for key, doc in KNOWLEDGE_BASE.items():
        if any(word in q for word in key.split("_")):
            return doc
    return "No relevant document found."

@observe()
def generate(query: str, context: str) -> str:
    """Generate an answer using Claude, grounded in the retrieved context."""
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer the question using only the context above. "
        "If the context doesn't contain the answer, say so."
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

@observe()
def rag_pipeline(query: str) -> str:
    """End-to-end: retrieve relevant context, then generate the answer."""
    context = retrieve(query)
    return generate(query, context)

In [6]:
demo_queries = [
    "How do I reset my VPN password?",
    "When are laptops refreshed?",
    "How does the vacation policy work?",
]

for q in demo_queries:
    print(f"Q: {q}")
    print(f"A: {rag_pipeline(q)}\n")

langfuse.flush()

Q: How do I reset my VPN password?
A: To reset your VPN password, visit selfservice.example.com and click 'Reset VPN'. The new password will be emailed to your registered address.

Q: When are laptops refreshed?
A: Based on the context:

- Standard laptops are refreshed every four years
- Specialized engineering workstations are refreshed every three years

So the answer depends on the type of laptop: standard laptops every four years, and specialized engineering workstations every three years.

Q: How does the vacation policy work?
A: Based on the context provided:

Full-time employees accrue 20 days of paid time off per year. After completing 5 years of service, this increases to 25 days per year.

However, the context doesn't provide additional details about how the vacation policy works, such as:
- How the time accrues (monthly, bi-weekly, etc.)
- Whether unused days roll over
- How to request time off
- Whether part-time employees receive PTO
- Blackout periods or other restrictio

---

## Scenario 3 — Dataset + experiment

A curated test set lives in Langfuse as a **Dataset**. Running the pipeline against it produces an **Experiment** — a named, versioned run that links each trace back to its source dataset item. This is the substrate for offline evaluation: every time you change the prompt, the retriever, or the model, you re-run the experiment and compare results.

**What to look for in the UI:** Navigate to **Datasets → internal-helpdesk-eval-v1**. The dataset items are listed, and the **Runs** tab shows the experiment we just executed, with each item's input, expected output, and the actual output side by side.

In [7]:
DATASET_NAME = "internal-helpdesk-eval-v1"

# Create the dataset (idempotent — safe to re-run).
langfuse.create_dataset(
    name=DATASET_NAME,
    description="Internal helpdesk FAQ evaluation set for the RAG showcase.",
)

items = [
    {"input": "How do I reset my VPN password?",
     "expected_output": "Visit selfservice.example.com, click 'Reset VPN'; the new password is emailed to you."},
    {"input": "When are laptops refreshed?",
     "expected_output": "Standard laptops every 4 years; engineering workstations every 3 years."},
    {"input": "How much vacation time do employees get?",
     "expected_output": "20 days per year, rising to 25 after 5 years of service."},
    {"input": "Where do I file an expense report?",
     "expected_output": "Concur portal at concur.example.com, within 30 days."},
    {"input": "How do I get guest WiFi access?",
     "expected_output": "Credentials at the reception desk; the network is 'Guest-Network'; rotates weekly."},
]

for item in items:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=item["input"],
        expected_output=item["expected_output"],
    )

print(f"Dataset '{DATASET_NAME}' created with {len(items)} items.")

Dataset 'internal-helpdesk-eval-v1' created with 5 items.


In [8]:
RUN_NAME = "rag-baseline-v1"

dataset = langfuse.get_dataset(DATASET_NAME)

def rag_task(*, item, **kwargs):
    """Experiment task — runs the pipeline on a single dataset item."""
    return rag_pipeline(item.input)

result = dataset.run_experiment(
    name=RUN_NAME,
    description="Baseline RAG pipeline run over the helpdesk eval set.",
    task=rag_task,
)

print(f"Experiment run '{result.run_name}' complete.")
if result.dataset_run_url:
    print("View it:", result.dataset_run_url)
langfuse.flush()

Experiment run 'rag-baseline-v1 - 2026-05-17T21:15:10.280517Z' complete.
View it: http://localhost:3000/project/cmpa567fs0006pi0751yay0nc/datasets/cmpa5dfni000cpi07ws68h5wq/runs/7d052cfd-94e3-4812-b129-520ad458ad6f


---

## Scenario 4 — LLM-as-judge faithfulness evaluator

Now attach a *score* to each trace using an LLM judge. The judge sees the question, the retrieved context, and the generated answer, and returns a faithfulness score (1–5) plus a one-sentence rationale.

**What to look for in the UI:** Open the experiment run **rag-baseline-v1-judged**. Each item now has a `faithfulness` score in the scores column, the rationale viewable in the trace detail, and the aggregated mean score visible at the experiment level.

In [9]:
import json

@observe()
def judge_faithfulness(question: str, context: str, answer: str) -> dict:
    """Score whether the answer is faithful to the retrieved context."""
    prompt = f"""You are evaluating whether an AI-generated answer is faithful to its source context.

Context:
{context}

Question:
{question}

Answer:
{answer}

Score on faithfulness from 1 to 5:
  5 — Every claim is directly supported by the context.
  4 — Almost all claims supported; minor reasonable inferences.
  3 — Most claims supported; some unsupported additions.
  2 — Significant unsupported claims.
  1 — Largely fabricated or contradicting the context.

Respond with ONLY a JSON object, no other text:
{{"score": <integer 1-5>, "rationale": "<one sentence>"}}"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()
    # Strip code fences if the model added them.
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

In [10]:
RUN_NAME_JUDGED = "rag-baseline-v1-judged"

dataset = langfuse.get_dataset(DATASET_NAME)

def rag_task(*, item, **kwargs):
    return rag_pipeline(item.input)

def faithfulness_evaluator(*, input, output, expected_output=None, **kwargs):
    """Re-retrieve context (cheap dict lookup) and ask the judge to score the answer."""
    context = retrieve(input)
    verdict = judge_faithfulness(input, context, output)
    return {
        "name": "faithfulness",
        "value": verdict["score"],
        "comment": verdict["rationale"],
    }

result = dataset.run_experiment(
    name=RUN_NAME_JUDGED,
    description="Baseline RAG with LLM-as-judge faithfulness scoring.",
    task=rag_task,
    evaluators=[faithfulness_evaluator],
)

def _get(ev, key):
    return ev.get(key) if isinstance(ev, dict) else getattr(ev, key, None)

for ir in result.item_results:
    faith = next((e for e in ir.evaluations if _get(e, "name") == "faithfulness"), None)
    score = _get(faith, "value") if faith else None
    rationale = _get(faith, "comment") if faith else ""
    print(f"Q: {ir.item.input}")
    print(f"A: {ir.output}")
    print(f"Faithfulness: {score}/5 — {rationale}\n")

if result.dataset_run_url:
    print("View it:", result.dataset_run_url)
langfuse.flush()

Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute 'name'
Evaluator failed: 'dict' object has no attribute

Q: How do I get guest WiFi access?
A: Based on the context provided, to get guest WiFi access you need to:

1. Go to the reception desk to obtain the WiFi credentials
2. Connect to the network named 'Guest-Network'
3. Use the credentials provided by reception (note that these credentials change weekly)
Faithfulness: 5/5 — Every claim in the answer is directly supported by the context, which states credentials are available at reception, the network name is 'Guest-Network', and credentials rotate weekly.

Q: Where do I file an expense report?
A: According to the context, you file expense reports through the Concur portal at concur.example.com.
Faithfulness: 5/5 — The answer directly restates information from the context about filing expense reports through the Concur portal at concur.example.com, with no unsupported claims or additions.

Q: How much vacation time do employees get?
A: According to the context, full-time employees get:
- 20 days of paid time off per year
- 25 days of paid

---

## Recap & one closing note

What we just demonstrated, in order:

1. **Tracing** — single API call → one trace with full request/response metadata.
2. **Hierarchy** — decorated functions → trace tree mirroring the pipeline structure.
3. **Datasets** — curated examples uploaded once → re-runnable experiments.
4. **Scoring** — LLM-as-judge attached to traces → quality metric per run, aggregated across runs.

### LangChain shortcut

If your pipeline is already built on LangChain or LangGraph, instrumenting it is a single line — Langfuse provides a callback handler you pass into the chain or graph:

```python
from langfuse.langchain import CallbackHandler
handler = CallbackHandler()

result = chain.invoke({"question": "..."}, config={"callbacks": [handler]})
```

Every chain step becomes an observation in the trace; the hierarchy is inferred from the chain structure. The trade-off versus the `@observe` approach in this notebook is that less of the structure is visible in your own code — useful for production, less instructive for a showcase.